# PricePulse — AI-Powered Real-Time Price Tracker

**Author: Shreyas**

---

## What This Notebook Does

PricePulse helps everyday shoppers avoid overpaying for products online. It:

1. Fetches **real, live prices** from Google Shopping (which aggregates Amazon, Walmart, Best Buy, eBay, Target, and more) via the SerpAPI `google_shopping` engine.
2. Stores every price lookup in a **local SQLite database** so price history accumulates over time.
3. Sends the price data and history to a **Groq-hosted LLM** (Llama 3.3 70B) to generate a structured buy/wait recommendation, factoring in price spreads, seasonal sale windows, and historical trends.
4. Displays a clean, readable price report with retailer breakdown and AI reasoning.

---

## Who It Is For

Anyone who wants data-driven guidance before making a purchase — no subscriptions, no paywalls, no credit card required for the APIs used here.

---

## Prerequisites

| Requirement | Notes |
|---|---|
| Python 3.8+ | Required for f-strings, type hints, and walrus operator support |
| SerpAPI free account | Sign up at [serpapi.com](https://serpapi.com) — 100 free searches/month, no credit card |
| Groq free account | Sign up at [console.groq.com](https://console.groq.com) — generous free tier, no credit card |
| `groq` Python package | Installed in Cell 1 below |
| `google-search-results` Python package | SerpAPI's official client, installed in Cell 1 below |

---

## API Keys

- **SerpAPI key**: After signing up at [serpapi.com](https://serpapi.com), find your key at [serpapi.com/manage-api-key](https://serpapi.com/manage-api-key).
- **Groq key**: After signing up at [console.groq.com](https://console.groq.com), create a key at [console.groq.com/keys](https://console.groq.com/keys).

Both are entered securely using `getpass` — they will never appear in your notebook output.

---

## Free Tier Limits

- **SerpAPI**: 100 searches/month. Each call to `hunt_price()` uses 1 search credit. Use thoughtfully.
- **Groq**: Rate limits are generous for personal use; the Llama 3.3 70B model is free with no credit card.


---
## Section 1 — Introduction & Setup

The cell below installs the two external packages this notebook requires.
All other dependencies (`requests`, `json`, `sqlite3`, `datetime`, `os`, `getpass`) are part of the Python standard library.

Run this cell once per environment. In Google Colab or a fresh virtual environment, it typically takes under 30 seconds.

In [1]:
# Install external dependencies
# groq          — official Groq Python client for LLM inference
# google-search-results — SerpAPI's official Python client
%pip install groq google-search-results --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Standard library imports
import json
import os
import sqlite3
from datetime import datetime
from getpass import getpass
from typing import Dict, List, Optional

# Third-party imports
from groq import Groq
from serpapi import GoogleSearch

print("All imports successful.")

All imports successful.


In [3]:
# Enter your SerpAPI key securely.
# Get yours (free, no credit card) at: https://serpapi.com/manage-api-key
SERPAPI_KEY: str = getpass("Enter your SerpAPI key: ")

if not SERPAPI_KEY:
    raise ValueError("SerpAPI key cannot be empty. Get a free key at https://serpapi.com/manage-api-key")

In [4]:
# Enter your Groq API key securely.
# Get yours (free, no credit card) at: https://console.groq.com/keys
GROQ_KEY: str = getpass("Enter your Groq API key: ")

if not GROQ_KEY:
    raise ValueError("Groq API key cannot be empty. Get a free key at https://console.groq.com/keys")

groq_client: Groq = Groq(api_key=GROQ_KEY)
print("Groq client initialized successfully.")

Groq client initialized successfully.


---
## Section 2 — Price History Database

### Why Persistent Price History Matters

A single price snapshot tells you the current price, but not whether it is cheap or expensive relative to the past.
By storing every price lookup in a local SQLite database, PricePulse builds a growing record that enables:

- **Trend analysis** — is the price rising or falling over time?
- **Better AI recommendations** — the LLM receives historical context, allowing it to identify genuine deals versus inflated prices before a holiday sale.
- **Session persistence** — the database survives notebook restarts, so your history accumulates across multiple days and weeks.

SQLite is ideal here: it is built into Python, requires no server, and produces a single portable `.db` file.
The database file is created in the same directory as this notebook the first time the schema cell is executed.

In [5]:
# Database path — co-located with this notebook for easy portability
DB_PATH: str = os.path.join(os.path.dirname(os.path.abspath("__file__")), "pricepulse_history.db")


def initialise_database(db_path: str) -> None:
    """Create the price_history table if it does not already exist.

    Parameters
    ----------
    db_path : str
        Absolute or relative path to the SQLite database file.
        The file is created automatically if it does not exist.

    Returns
    -------
    None
    """
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()
    cursor.execute(
        """
        CREATE TABLE IF NOT EXISTS price_history (
            id            INTEGER PRIMARY KEY AUTOINCREMENT,
            product_name  TEXT    NOT NULL,
            retailer      TEXT    NOT NULL,
            price         REAL    NOT NULL,
            currency      TEXT    NOT NULL DEFAULT 'USD',
            timestamp     TEXT    NOT NULL
        )
        """
    )
    connection.commit()
    connection.close()


initialise_database(DB_PATH)
print(f"Database ready at: {DB_PATH}")

Database ready at: /Users/abhisheklonkar/Desktop/pricepulse-ai/pricepulse_history.db


In [6]:
def save_price_record(
    product_name: str,
    retailer: str,
    price: float,
    currency: str = "USD",
    db_path: str = DB_PATH,
) -> None:
    """Persist a single price observation to the SQLite database.

    Parameters
    ----------
    product_name : str
        Human-readable product search term, e.g. "Sony WH-1000XM5".
    retailer : str
        Name of the retailer from the shopping result, e.g. "Amazon".
    price : float
        Numeric price in the specified currency.
    currency : str
        ISO 4217 currency code; defaults to "USD".
    db_path : str
        Path to the SQLite database file.

    Returns
    -------
    None
    """
    timestamp = datetime.now().isoformat()
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()
    cursor.execute(
        """
        INSERT INTO price_history (product_name, retailer, price, currency, timestamp)
        VALUES (?, ?, ?, ?, ?)
        """,
        (product_name.lower().strip(), retailer, price, currency, timestamp),
    )
    connection.commit()
    connection.close()


def get_price_history(
    product_name: str,
    limit: int = 50,
    db_path: str = DB_PATH,
) -> List[Dict]:
    """Retrieve the most recent price observations for a product.

    Parameters
    ----------
    product_name : str
        Product search term; matched case-insensitively.
    limit : int
        Maximum number of rows to return, ordered newest-first.
    db_path : str
        Path to the SQLite database file.

    Returns
    -------
    List[Dict]
        List of dicts with keys: retailer, price, currency, timestamp.
        Returns an empty list if no records exist.
    """
    connection = sqlite3.connect(db_path)
    cursor = connection.cursor()
    cursor.execute(
        """
        SELECT retailer, price, currency, timestamp
        FROM price_history
        WHERE LOWER(product_name) = ?
        ORDER BY timestamp DESC
        LIMIT ?
        """,
        (product_name.lower().strip(), limit),
    )
    rows = cursor.fetchall()
    connection.close()
    return [
        {"retailer": row[0], "price": row[1], "currency": row[2], "timestamp": row[3]}
        for row in rows
    ]


print("Database utility functions defined: save_price_record, get_price_history")

Database utility functions defined: save_price_record, get_price_history


---
## Section 3 — Fetching Real Prices via SerpAPI

### Google Shopping as a Data Source

Rather than scraping individual retailer websites (which is fragile, easily blocked, and legally murky),
PricePulse uses the **SerpAPI `google_shopping` engine** to query Google Shopping.

Google Shopping already aggregates listings from:
Amazon, Walmart, Best Buy, eBay, Target, Newegg, B&H Photo, Costco, and hundreds of other merchants — all in a single API call.

### What the API Returns

Each item in the `shopping_results` array includes:

| Field | Description |
|---|---|
| `title` | Product listing title |
| `price` | Price as a formatted string, e.g. `"$299.99"` |
| `extracted_price` | Price as a float (already parsed by SerpAPI) |
| `source` | Retailer name |
| `link` | Direct URL to the product listing |

### Free Tier Efficiency

SerpAPI provides 100 free searches per month (no credit card). Each `hunt_price()` call consumes exactly 1 credit.
To conserve credits during development, run the demo section only when you need live data.
For scheduled monitoring, consider running this notebook once per day rather than once per hour.

In [7]:
def fetch_google_shopping_prices(
    product_name: str,
    serpapi_key: str,
) -> List[Dict]:
    """Fetch real-time product prices from Google Shopping via SerpAPI.

    Uses the official SerpAPI `google_shopping` engine, which aggregates
    listings from Amazon, Walmart, Best Buy, eBay, Target, and others.
    No mock data is returned under any circumstances; if no results are
    found, a descriptive ValueError is raised.

    Parameters
    ----------
    product_name : str
        The search query, e.g. "Sony WH-1000XM5 headphones".
    serpapi_key : str
        A valid SerpAPI API key with remaining search credits.

    Returns
    -------
    List[Dict]
        List of dicts, each with keys:
        - retailer (str): Store name from Google Shopping
        - price (float): Numeric price in the listing currency
        - currency (str): Currency code inferred from the price string
        - title (str): Product title as listed
        - link (str): URL to the product listing

    Raises
    ------
    ValueError
        If the API call succeeds but returns zero shopping results.
    RuntimeError
        If SerpAPI returns an error field in the response.
    """
    search_params = {
        "engine": "google_shopping",
        "q": product_name,
        "api_key": serpapi_key,
        "num": 20,
    }

    search = GoogleSearch(search_params)
    response = search.get_dict()

    # SerpAPI surfaces API-level errors in an "error" key
    if "error" in response:
        raise RuntimeError(
            f"SerpAPI returned an error for query '{product_name}': {response['error']}. "
            "Check that your API key is valid and that you have remaining search credits."
        )

    shopping_results: List[Dict] = response.get("shopping_results", [])

    if not shopping_results:
        raise ValueError(
            f"Google Shopping returned no results for '{product_name}'. "
            "Try a more specific or more common product name."
        )

    parsed_results: List[Dict] = []
    for item in shopping_results:
        extracted_price = item.get("extracted_price")
        if extracted_price is None:
            # Skip listings with no numeric price (e.g. "Call for price")
            continue

        parsed_results.append(
            {
                "retailer": item.get("source", "Unknown Retailer"),
                "price": float(extracted_price),
                "currency": "USD",  # Google Shopping defaults to USD; add `gl` param to localise
                "title": item.get("title", product_name),
                "link": item.get("link", ""),
            }
        )

    if not parsed_results:
        raise ValueError(
            f"Google Shopping returned {len(shopping_results)} listings for '{product_name}', "
            "but none had a parseable numeric price. This is unusual — please try again."
        )

    return parsed_results


print("fetch_google_shopping_prices defined.")

fetch_google_shopping_prices defined.


In [8]:
def display_prices_table(results: List[Dict]) -> None:
    """Print a formatted price comparison table to stdout.

    Outputs columns for rank, retailer, price, and currency using
    only the Python standard library (no pandas or tabulate required).
    Results are sorted by ascending price so the best deal appears first.

    Parameters
    ----------
    results : List[Dict]
        Output of fetch_google_shopping_prices(); each dict must contain
        'retailer', 'price', and 'currency' keys.

    Returns
    -------
    None
    """
    if not results:
        print("No price data to display.")
        return

    sorted_results = sorted(results, key=lambda item: item["price"])

    header = f"{'#':<4} {'Retailer':<30} {'Price':>10}  {'Currency'}"
    separator = "-" * len(header)

    print(separator)
    print(header)
    print(separator)

    for rank, item in enumerate(sorted_results, start=1):
        print(
            f"{rank:<4} {item['retailer']:<30} {item['price']:>10.2f}  {item['currency']}"
        )

    print(separator)
    best = sorted_results[0]
    print(
        f"Best deal: {best['retailer']} at {best['currency']} {best['price']:.2f}"
    )


print("display_prices_table defined.")

display_prices_table defined.


---
## Section 4 — AI Buying Recommendation Engine

### Why Use an LLM for Buy/Wait Decisions?

A purely rule-based approach — "buy if the current price is below the 30-day average" — misses crucial context:

- **Seasonal patterns**: Black Friday (late November), Prime Day (July), back-to-school (August), and post-holiday clearance (January) all create predictable dip windows that a calendar-aware LLM can reason about.
- **Price spread interpretation**: A wide spread between the cheapest and most expensive listing may signal a temporarily discounted item or a grey-market seller, which requires qualitative judgment.
- **Risk vs. savings tradeoff**: If the potential savings by waiting is only $3 on a $300 item, the risk of stockouts or price rises may not be worth it. An LLM can weigh these tradeoffs contextually.

The LLM is instructed to respond exclusively in valid JSON, making the output deterministic and machine-parseable across every run.
The `response_format={"type": "json_object"}` parameter enforces this at the API level.

In [9]:
def build_analysis_prompt(
    product: str,
    price_data: List[Dict],
    history: List[Dict],
) -> str:
    """Construct a structured prompt for the LLM buying-recommendation analysis.

    The prompt summarises current price statistics, historical price context,
    and the current date, then instructs the model to respond in strict JSON.

    Parameters
    ----------
    product : str
        The product search term.
    price_data : List[Dict]
        Current price listings from fetch_google_shopping_prices().
    history : List[Dict]
        Historical records from get_price_history(); may be empty.

    Returns
    -------
    str
        A fully formatted prompt string ready to send to the LLM.
    """
    prices = [item["price"] for item in price_data]
    min_price = min(prices)
    max_price = max(prices)
    avg_price = sum(prices) / len(prices)
    price_spread = max_price - min_price

    retailer_summary = "\n".join(
        f"  - {item['retailer']}: {item['currency']} {item['price']:.2f}"
        for item in sorted(price_data, key=lambda x: x["price"])[:8]
    )

    if history:
        historical_prices = [record["price"] for record in history]
        history_summary = (
            f"  Historical min: USD {min(historical_prices):.2f}\n"
            f"  Historical avg: USD {sum(historical_prices)/len(historical_prices):.2f}\n"
            f"  Historical max: USD {max(historical_prices):.2f}\n"
            f"  Records available: {len(historical_prices)}"
        )
    else:
        history_summary = "  No historical data yet (first lookup for this product)."

    current_date = datetime.now().strftime("%B %d, %Y")

    prompt = f"""You are a professional pricing analyst helping a consumer decide whether to buy a product now or wait for a better price.

Product: {product}
Date of analysis: {current_date}

Current market prices (sorted lowest to highest):
{retailer_summary}

Current price statistics:
  Lowest price: USD {min_price:.2f}
  Average price: USD {avg_price:.2f}
  Highest price: USD {max_price:.2f}
  Price spread: USD {price_spread:.2f}

Price history from previous lookups:
{history_summary}

Analysis factors to consider:
1. Is the current lowest price a genuine deal compared to the average and historical data?
2. Are there upcoming sale events (Black Friday, Prime Day, back-to-school, post-holiday clearance) that could yield a lower price?
3. Is the price spread wide enough to suggest a temporarily discounted listing?
4. What is the risk of waiting — could prices rise, or is the product likely to be discounted further?

Respond ONLY with a valid JSON object using exactly these fields:
{{
  "recommendation": "buy_now" | "wait" | "set_alert",
  "confidence": <integer 0-100>,
  "reasoning": "<one to two sentence plain-English explanation>",
  "predicted_best_price": <numeric float, your estimate of the best achievable price in the next 30-60 days>,
  "predicted_best_time": "<description of the best time window to buy, e.g. 'Late November 2025 (Black Friday)'>",
  "savings_potential": <numeric float, estimated savings versus current lowest price by waiting>
}}"""

    return prompt


print("build_analysis_prompt defined.")

build_analysis_prompt defined.


In [10]:
def get_ai_recommendation(
    prompt: str,
    groq_client: Groq,
    model: str = "llama-3.3-70b-versatile",
) -> Dict:
    """Call the Groq LLM and parse the structured buy/wait recommendation.

    Sends the analysis prompt to Groq's Llama 3.3 70B model with
    `response_format={"type": "json_object"}` enforced at the API level,
    ensuring the output is always valid JSON.

    Parameters
    ----------
    prompt : str
        The fully formatted prompt from build_analysis_prompt().
    groq_client : Groq
        An initialised Groq client with a valid API key.
    model : str
        The Groq model identifier; defaults to llama-3.3-70b-versatile.

    Returns
    -------
    Dict
        Parsed JSON recommendation with keys: recommendation, confidence,
        reasoning, predicted_best_price, predicted_best_time, savings_potential.

    Raises
    ------
    RuntimeError
        If the Groq API call fails or the response cannot be parsed as JSON.
    """
    try:
        response = groq_client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a pricing analyst. You always respond with valid JSON only. "
                        "Do not include any text before or after the JSON object."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            temperature=0.2,
            max_tokens=400,
            response_format={"type": "json_object"},
        )
        raw_content: str = response.choices[0].message.content
        return json.loads(raw_content)

    except json.JSONDecodeError as parse_error:
        raise RuntimeError(
            f"Groq returned a response that could not be parsed as JSON: {parse_error}. "
            "This is unexpected when response_format is set to json_object — please try again."
        ) from parse_error

    except Exception as api_error:
        raise RuntimeError(
            f"Groq API call failed: {api_error}. "
            "Verify your API key at https://console.groq.com/keys and check your rate limits."
        ) from api_error


print("get_ai_recommendation defined.")

get_ai_recommendation defined.


---
## Section 5 — Price Drop Hunter: Full Pipeline

This section assembles the complete end-to-end pipeline.

A single call to `hunt_price()` will:
1. Query Google Shopping via SerpAPI for real-time prices.
2. Save every returned price record to the SQLite database.
3. Retrieve any existing price history for the product from the database.
4. Build and send an analysis prompt to the Groq LLM.
5. Return a structured result dictionary containing prices, best deal, history, and the AI recommendation.

The `display_full_report()` function formats this result into a clean, human-readable report.

In [11]:
def hunt_price(
    product_name: str,
    serpapi_key: str,
    groq_client: Groq,
    target_price: Optional[float] = None,
) -> Dict:
    """Orchestrate the full PricePulse pipeline for a single product.

    Fetches real-time prices from Google Shopping, persists them to the
    local SQLite database, retrieves price history, and calls the LLM
    for a buy/wait recommendation.

    Parameters
    ----------
    product_name : str
        The product search term, e.g. "Sony WH-1000XM5".
    serpapi_key : str
        A valid SerpAPI API key.
    groq_client : Groq
        An initialised Groq client.
    target_price : Optional[float]
        The user's desired maximum price. Used for alert reporting only;
        does not affect the AI analysis.

    Returns
    -------
    Dict
        A result dict with keys:
        - product (str): Search term
        - timestamp (str): ISO timestamp of this run
        - target_price (Optional[float]): User's target price
        - price_data (List[Dict]): All price listings from Google Shopping
        - best_deal (Dict): The single lowest-priced listing
        - history (List[Dict]): Historical records from the database
        - ai_recommendation (Dict): Parsed LLM recommendation
    """
    print(f"Fetching prices for: {product_name}")

    # Step 1: Fetch live prices (raises on no results — no silent fallback)
    price_data = fetch_google_shopping_prices(product_name, serpapi_key)
    print(f"  Found {len(price_data)} listings from Google Shopping.")

    # Step 2: Persist all returned listings to the database
    for listing in price_data:
        save_price_record(
            product_name=product_name,
            retailer=listing["retailer"],
            price=listing["price"],
            currency=listing["currency"],
        )
    print(f"  Saved {len(price_data)} records to the price history database.")

    # Step 3: Retrieve accumulated price history for richer LLM context
    history = get_price_history(product_name, limit=50)
    print(f"  Retrieved {len(history)} historical price records.")

    # Step 4: Identify the best deal (lowest price)
    best_deal = min(price_data, key=lambda item: item["price"])

    # Step 5: Build the prompt and get the AI recommendation
    print("  Requesting AI recommendation from Groq...")
    prompt = build_analysis_prompt(product_name, price_data, history)
    ai_recommendation = get_ai_recommendation(prompt, groq_client)
    print("  AI recommendation received.")

    return {
        "product": product_name,
        "timestamp": datetime.now().isoformat(),
        "target_price": target_price,
        "price_data": price_data,
        "best_deal": best_deal,
        "history": history,
        "ai_recommendation": ai_recommendation,
    }


print("hunt_price defined.")

hunt_price defined.


In [12]:
def display_full_report(result: Dict) -> None:
    """Print a clean, professional price analysis report to stdout.

    Formats and displays all fields from the hunt_price() result dict,
    including best deal, retailer breakdown, AI recommendation, and an
    optional target price alert.

    Parameters
    ----------
    result : Dict
        The structured result dict returned by hunt_price().

    Returns
    -------
    None
    """
    product = result["product"]
    best = result["best_deal"]
    rec = result["ai_recommendation"]
    target = result["target_price"]
    price_data = result["price_data"]
    timestamp = result["timestamp"]

    separator_thick = "=" * 64
    separator_thin = "-" * 64

    print(separator_thick)
    print(f"  PRICE REPORT: {product.upper()}")
    print(f"  Analysis timestamp: {timestamp}")
    print(separator_thick)

    # Best deal
    print(f"\n  BEST DEAL: {best['currency']} {best['price']:.2f} at {best['retailer']}")
    if target is not None:
        if best["price"] <= target:
            print(
                f"  TARGET PRICE ALERT: Current best deal ({best['currency']} {best['price']:.2f}) "
                f"is at or below your target of {best['currency']} {target:.2f}."
            )
        else:
            diff = best["price"] - target
            print(
                f"  Not yet at target: {best['currency']} {diff:.2f} above your "
                f"target of {best['currency']} {target:.2f}."
            )

    # Retailer breakdown
    print(f"\n  RETAILER BREAKDOWN ({len(price_data)} listings):")
    print(separator_thin)
    for item in sorted(price_data, key=lambda x: x["price"])[:10]:
        print(f"    {item['retailer']:<28} {item['currency']} {item['price']:.2f}")

    # AI recommendation
    print(f"\n  AI RECOMMENDATION")
    print(separator_thin)
    action = rec.get("recommendation", "N/A").upper().replace("_", " ")
    confidence = rec.get("confidence", "N/A")
    reasoning = rec.get("reasoning", "N/A")
    best_price = rec.get("predicted_best_price", "N/A")
    best_time = rec.get("predicted_best_time", "N/A")
    savings = rec.get("savings_potential", "N/A")

    print(f"    Action:             {action}")
    print(f"    Confidence:         {confidence}%")
    print(f"    Reasoning:          {reasoning}")
    print(f"    Predicted best price: USD {best_price}")
    print(f"    Best time to buy:   {best_time}")
    print(f"    Savings potential:  USD {savings}")
    print(separator_thick)
    print()


print("display_full_report defined.")

display_full_report defined.


---
## Section 6 — Interactive Demo

### Running the Demo

The cells below run PricePulse on three real consumer products and then open
an interactive prompt where you can enter any product and an optional target price.

**Before running:**
- Confirm that both API keys have been entered in Section 1.
- Each product in the demo below consumes **1 SerpAPI credit**.
  With the free tier of 100/month, running the three-product demo uses 3 credits total.

**Interpreting the output:**
- `BUY NOW` — the current price is a strong deal; act promptly.
- `WAIT` — better pricing is expected; set a reminder for the predicted window.
- `SET ALERT` — the price is borderline; consider checking again in a few days.

In [13]:
# Demo: three real consumer products
# Change these search terms to any products you want to evaluate.
DEMO_PRODUCTS = [
    {"name": "Sony WH-1000XM5",  "target_price": 280.0},
    {"name": "iPad Air M2",       "target_price": 550.0},
    {"name": "Dyson V15 Detect",  "target_price": 500.0},
]

demo_results = []

for demo_product in DEMO_PRODUCTS:
    print(f"\nSearching for: {demo_product['name']}")
    print("Please wait...")
    result = hunt_price(
        product_name=demo_product["name"],
        serpapi_key=SERPAPI_KEY,
        groq_client=groq_client,
        target_price=demo_product["target_price"],
    )
    demo_results.append(result)
    display_full_report(result)


Searching for: Sony WH-1000XM5
Please wait...
Fetching prices for: Sony WH-1000XM5
  Found 22 listings from Google Shopping.
  Saved 22 records to the price history database.
  Retrieved 22 historical price records.
  Requesting AI recommendation from Groq...
  AI recommendation received.
  PRICE REPORT: SONY WH-1000XM5
  Analysis timestamp: 2026-03-04T00:33:32.275238

  BEST DEAL: USD 111.80 at politimetro.com.br
  TARGET PRICE ALERT: Current best deal (USD 111.80) is at or below your target of USD 280.00.

  RETAILER BREAKDOWN (22 listings):
----------------------------------------------------------------
    politimetro.com.br           USD 111.80
    iTabPro                      USD 123.00
    sewanaka.net                 USD 155.00
    location-en-corse.fr         USD 198.00
    Woot                         USD 227.99
    techtitanlb.com              USD 245.00
    Cliq                         USD 290.00
    SPEX                         USD 300.00
    selfridges.com              

In [14]:
# Interactive product tracker
# Enter a product name, then optionally enter a target price.
# Type 'quit' or 'exit' to stop.

print("PricePulse Interactive Tracker")
print("=" * 40)
print("Type a product name to search, or 'quit' to exit.")
print()

while True:
    user_product = input("Product to track: ").strip()

    if user_product.lower() in ("quit", "exit", "q"):
        print("Exiting interactive tracker.")
        break

    if not user_product:
        print("Please enter a product name.")
        continue

    raw_target = input("Target price in USD (press Enter to skip): ").strip()
    user_target: Optional[float] = None

    if raw_target:
        try:
            user_target = float(raw_target)
        except ValueError:
            print(f"Could not parse '{raw_target}' as a price — target ignored.")

    print(f"\nSearching for: {user_product}")
    print("Please wait...\n")

    try:
        user_result = hunt_price(
            product_name=user_product,
            serpapi_key=SERPAPI_KEY,
            groq_client=groq_client,
            target_price=user_target,
        )
        display_full_report(user_result)
    except (ValueError, RuntimeError) as search_error:
        print(f"Search failed: {search_error}")
        print("Please try a different product name.")

PricePulse Interactive Tracker
Type a product name to search, or 'quit' to exit.


Searching for: ipad M2
Please wait...

Fetching prices for: ipad M2
  Found 40 listings from Google Shopping.
  Saved 40 records to the price history database.
  Retrieved 40 historical price records.
  Requesting AI recommendation from Groq...
  AI recommendation received.
  PRICE REPORT: IPAD M2
  Analysis timestamp: 2026-03-04T00:36:05.670812

  BEST DEAL: USD 36.00 at Só Ombrelones
  TARGET PRICE ALERT: Current best deal (USD 36.00) is at or below your target of USD 800.00.

  RETAILER BREAKDOWN (40 listings):
----------------------------------------------------------------
    Só Ombrelones                USD 36.00
    eBay - vc_tech_services      USD 39.50
    Só Ombrelones                USD 42.00
    Só Ombrelones                USD 74.00
    eBay - htam86                USD 199.00
    eBay - aasieltd              USD 199.99
    eBay                         USD 239.00
    eBay - itsworthmore     

---
## Section 7 — Price History Analysis

### Why Review Historical Price Data?

The more runs you complete, the more price history accumulates in the database.
Over days and weeks, this history allows you to:

- **Validate AI recommendations** — did the model correctly predict a price drop, or did prices rise instead?
- **Spot slow trends** — a product rising $5/month is on a very different trajectory than one that is stable.
- **Identify seasonal dips** — checking your history around key sale events shows whether the discounts were meaningful.

The function below queries the database directly and prints a timestamped log for any product you have previously tracked.

In [15]:
def show_price_history(product_name: str, limit: int = 30) -> None:
    """Print a timestamped price history log for a tracked product.

    Queries the local SQLite database and displays all stored price
    observations for the given product, newest first, up to `limit` rows.

    Parameters
    ----------
    product_name : str
        The product search term used when calling hunt_price().
    limit : int
        Maximum number of records to display; defaults to 30.

    Returns
    -------
    None
    """
    records = get_price_history(product_name, limit=limit)

    separator = "-" * 64
    print(separator)
    print(f"  Price History: {product_name}")
    print(f"  Showing up to {limit} most recent records")
    print(separator)

    if not records:
        print(
            f"  No price history found for '{product_name}'.\n"
            "  Run hunt_price() for this product first to start building history."
        )
        print(separator)
        return

    header = f"  {'Timestamp':<26} {'Retailer':<28} {'Price':>8}  {'Currency'}"
    print(header)
    print(separator)

    for record in records:
        # Trim ISO timestamp to second precision for readability
        readable_timestamp = record["timestamp"][:19].replace("T", " ")
        print(
            f"  {readable_timestamp:<26} {record['retailer']:<28} "
            f"{record['price']:>8.2f}  {record['currency']}"
        )

    prices_only = [record["price"] for record in records]
    print(separator)
    print(
        f"  Summary over {len(records)} records: "
        f"min USD {min(prices_only):.2f}  "
        f"avg USD {sum(prices_only)/len(prices_only):.2f}  "
        f"max USD {max(prices_only):.2f}"
    )
    print(separator)


# Example: view history for a product searched in Section 6
# Change this to any product you have previously tracked.
show_price_history("Sony WH-1000XM5")

----------------------------------------------------------------
  Price History: Sony WH-1000XM5
  Showing up to 30 most recent records
----------------------------------------------------------------
  Timestamp                  Retailer                        Price  Currency
----------------------------------------------------------------
  2026-03-04 00:36:39        wafuu.com                      462.00  USD
  2026-03-04 00:36:39        wafuu.com                      462.00  USD
  2026-03-04 00:36:39        wafuu.com                      462.00  USD
  2026-03-04 00:36:39        wafuu.com                      462.00  USD
  2026-03-04 00:36:39        location-en-corse.fr           198.00  USD
  2026-03-04 00:36:39        nhkseating.com               18920.00  USD
  2026-03-04 00:36:39        Amphasis Design               7804.65  USD
  2026-03-04 00:36:39        politimetro.com.br             111.80  USD
  2026-03-04 00:36:39        wafuu.com                      462.00  USD
  2026-0

---
## Section 8 — Limitations & Next Steps

### Current Limitations

**SerpAPI free tier (100 searches/month)**
Each call to `hunt_price()` consumes one search credit. At 100 credits/month, this is well-suited to personal tracking of a handful of products but is not appropriate for high-frequency monitoring of large watchlists without upgrading to a paid plan.

**Regional variation in Google Shopping results**
Results default to the US market. SerpAPI supports `gl` (country) and `hl` (language) parameters in the search query if you need localised prices. Add these to the `search_params` dict inside `fetch_google_shopping_prices()` to customise for your region.

**Price history improves over time**
The SQLite database only contains records from runs you have performed. On the first run for any product, the AI has no historical baseline and must reason solely from the current listings. The more frequently you run the notebook over days and weeks, the more context the AI receives and the more accurate its recommendations become.

**No real-time alerting baked in**
This notebook does not monitor prices continuously — it is a run-on-demand tool. Continuous monitoring requires a scheduler (see Next Steps below).

---

### Suggested Next Steps

1. **Schedule runs with `cron` or GitHub Actions**
   Run this notebook automatically once per day using `papermill` and a cron job or a GitHub Actions workflow. This ensures price history accumulates without manual effort and gives the AI a richer dataset to work with over time.

2. **Add email or SMS alerts**
   Use Python's `smtplib` (for email) or the Twilio free tier (for SMS) to send a notification when a product's price drops below your target. Integrate this into `display_full_report()` conditionally when a target price is set.

3. **Maintain a watchlist file**
   Create a `watchlist.json` file with product names and target prices. Have the notebook read and iterate over this file at startup, so tracking new products requires only editing JSON rather than modifying code.

4. **Extend to other markets**
   Add the `gl` and `hl` SerpAPI parameters to support UK (gbp), EU (eur), or other regional price lookups. Store the currency in the SQLite database (already supported) and display it in reports.

5. **Add a price alert dashboard**
   Use `matplotlib` (standard in most Python environments) to chart price history over time for tracked products, giving a visual representation of price trends alongside the AI's textual reasoning.
